In [1]:
# Now to download population projections by SA2 District 5 year projections

import geopandas as gpd
from pathlib import Path
import pandas as pd
import requests
import openpyxl

In [ ]:
Current_Path = Path.cwd()
Root_Path = Current_Path.parent.parent

Data_Path = Root_Path/"data"/"forecasting"
File_Path = Data_Path/"SAP_Forecast_2136.xlsx"

Data_Path.mkdir(parents=True, exist_ok=True)

# Now to read in the data

Uncleaned_Males = pd.read_excel(File_Path,
                               sheet_name="ERP_by_Age_Males",
                               header=None
                               )

Uncleaned_Females = pd.read_excel(File_Path,
                               sheet_name="ERP_by_Age_Females",
                               header=None
                               )

Uncleaned_Persons = pd.read_excel(File_Path,
                               sheet_name="ERP_by_Age_Persons",
                               header=None
                               )

In [3]:

###############################################

# Now to perform some preliminary data cleaning

###############################################

def clean_age_data(Data, Sex):

    # Get Years and Age Groups
    Years = Data.iloc[8].copy()
    Headers = Data.iloc[9].copy()
 
    Column_Names = [
        "GCCSA",
        "SA4_Code",
        "SA3_Code",
        "SA2_Code",
        "Region_Type",
        "Region",
    ]

    for i in range(6, len(Data.columns)):

        Year = Years.iloc[i]

        Age_Group = str(
            Headers.iloc[i]
        )

        # Clean age-group names.
        Age_Group = (
            Age_Group
            .replace("\n", " ")
            .replace(f"{Sex}s ", "")
            .strip()
            .replace(" ", "_")
        )

        Column_Names.append(
            f"{int(Year)}_{Age_Group}"
        )

    # Remove Metadata Rows
  

    Data = Data.iloc[11:].copy()

    Data.columns = Column_Names

    # Keep SA2 Areas Only

    Data = Data[
        Data["Region_Type"] == "SA2"
    ].copy()

    Data["SA2_Code"] = (
        Data["SA2_Code"]
        .astype(str)
    )

    Population_Columns = Data.columns[6:]

    Data[Population_Columns] = (
        Data[Population_Columns]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )
    # Add Sex Column

    Data["Sex"] = Sex

    Data = Data.reset_index(
        drop=True
    )

    return Data

In [4]:
# Now to clean my dataframe :)

Male_Data = clean_age_data(Uncleaned_Males, "Males")
Female_Data = clean_age_data(Uncleaned_Females, "Females")
Person_Data = clean_age_data(Uncleaned_Persons, "Persons")

In [5]:
#### Now to manipulate data to make it merge the way i want it ####

ID_Columns = [
    "GCCSA",
    "SA4_Code",
    "SA3_Code",
    "SA2_Code",
    "Region_Type",
    "Region",
]

Male_Data = Male_Data.rename(
    columns={
        Column: f"Male_{Column}"
        for Column in Male_Data.columns
        if Column not in ID_Columns
        and Column != "Sex"
    }
)

Female_Data = Female_Data.rename(
    columns={
        Column: f"Female_{Column}"
        for Column in Female_Data.columns
        if Column not in ID_Columns
        and Column != "Sex"
    }
)

Person_Data = Person_Data.rename(
    columns={
        Column: f"Persons_{Column}"
        for Column in Person_Data.columns
        if Column not in ID_Columns
        and Column != "Sex"
    }
)

In [6]:
# Now to drop sex coloum since i dont need it

Male_Data = Male_Data.drop(
    columns=["Sex"],
    errors="ignore",
)

Female_Data = Female_Data.drop(
    columns=["Sex"],
    errors="ignore",
)

Person_Data = Person_Data.drop(
    columns=["Sex"],
    errors="ignore",
)

In [7]:
# Now to merge my data into one :)

Clean_Data = Male_Data.merge(
    Female_Data.drop(
        columns=[
            "GCCSA",
            "SA4_Code",
            "SA3_Code",
            "Region_Type",
            "Region",
        ],
    ),
    on="SA2_Code",
    how="inner",
)

Clean_Data = Clean_Data.merge(
    Person_Data.drop(
        columns=[
            "GCCSA",
            "SA4_Code",
            "SA3_Code",
            "Region_Type",
            "Region",
        ],
    ),
    on="SA2_Code",
    how="inner",
)

In [8]:
# Now just to clean up the coloumn names

Clean_Data.columns = (
    Clean_Data.columns
    .str.replace("_Males__", "_", regex=False)
    .str.replace("_Females__", "_", regex=False)
    .str.replace("_Persons__", "_", regex=False)
)

In [9]:
Csv_File_Path = Data_Path/"SAP_Forecast_2136_Cleaned.csv"

Clean_Data.to_csv(Csv_File_Path, index=False)